# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert metadata to dict for printing
metadata = dataset.metadata.to_json()
print(f"Dataset loaded: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This will help you understand the structure of the dataset. All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets along with their @id and names
print("Available Record Sets (by @id):")
for rset in dataset.record_sets():
    print(f"- @id: {rset.id} | name: {rset.name}")

# For demonstration, we'll list the fields in all record sets
print('\nFields in each record set:')
for rset in dataset.record_sets():
    print(f"\nRecord set @id: {rset.id}, name: {rset.name}")
    print("Fields:")
    for field in rset.fields:
        print(f"  - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

*For this dataset, we'll load all record sets as DataFrames. Replace the record set IDs and field IDs with the ones relevant to your exploration.*

In [ ]:
# Choose all record sets (using their @ids)
record_sets = [rset.id for rset in dataset.record_sets()]
dataframes = {}

print(f"Extracting data for {len(record_sets)} record set(s)...\n")

for record_set_id in record_sets:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set @id: {record_set_id} => Shape: {df.shape}, columns: {df.columns.tolist()}")

# Select the main survey record set for further work
# Replace with the correct record set '@id' if needed
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"\nShowing first 5 rows for record set @id: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. The code below demonstrates how to select a field by `@id` and perform some basic EDA.

In [ ]:
# Let's demonstrate EDA on a numeric field (e.g., PHQ-9 Score) if present
# You'll need to use the @id of the field; replace if needed

df = dataframes[main_record_set_id].copy()

# List available columns for the chosen record set
print(f"Available columns (fields, by @id):\n{df.columns.tolist()}")

# Try to find a numeric field (e.g., phq9_score or similar)

possible_numeric_fields = [col for col in df.columns if ('phq' in col.lower() or 'gad' in col.lower() or 'score' in col.lower() or 'age' in col.lower())]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Selected numeric field for EDA: {numeric_field}\n")
else:
    # If not found, use the first column as fallback
    numeric_field = df.columns[0]
    print(f"Selected first field (fallback): {numeric_field}\n")

# Remove non-numeric or missing rows for this field
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > 10]
print(f"Filtered records with {numeric_field} > 10:")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely demographic field (e.g., gender, education, etc, by @id)
group_fields = [col for col in df.columns if ('gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower())]
if group_fields:
    group_field = group_fields[0]
    print(f"\nGrouped statistics by {group_field}:\n")
    grouped = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
    display(grouped)
else:
    print("No suitable group field detected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of the selected numeric field and a boxplot grouped by a demographic variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

# Boxplot by group field (if detected previously)
if group_fields:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to load the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, explore its structure, and perform basic exploratory data analysis and visualization. All references to dataset structure (record sets, fields, columns) were made using their `@id`.

You can extend this notebook to deeper analyses, feature engineering, or building machine learning models based on the dataset.